In [ ]:
import numpy as np


def effective_smoothing_length(
    window_length_ns,
    n_smoothings,
    sampling_period_ns=10.0,
):
    """
    Convert repeated moving-average smoothing into an equivalent
    single-window length, matched by variance.

    Parameters
    ----------
    window_length_ns : float
        Requested width of each moving-average window in ns.
    n_smoothings : int
        Number of times the moving average is applied.
    sampling_period_ns : float
        Time between samples in ns. For 100 MHz sampling, use 10 ns.

    Returns
    -------
    dict
        Window size, effective width, total support and discretization
        information.
    """
    if window_length_ns <= 0:
        raise ValueError("window_length_ns must be positive.")

    if n_smoothings < 1 or int(n_smoothings) != n_smoothings:
        raise ValueError("n_smoothings must be a positive integer.")

    if sampling_period_ns <= 0:
        raise ValueError("sampling_period_ns must be positive.")

    n_smoothings = int(n_smoothings)

    # Hardware/software filter requires an integer number of samples
    window_samples = max(
        1,
        int(np.rint(window_length_ns / sampling_period_ns)),
    )

    actual_window_ns = window_samples * sampling_period_ns

    # Equal-variance equivalent boxcar
    effective_samples = np.sqrt(
        n_smoothings * (window_samples**2 - 1) + 1
    )
    effective_length_ns = effective_samples * sampling_period_ns

    # Exact nonzero support of the convolved discrete kernel
    support_samples = n_smoothings * (window_samples - 1) + 1
    support_span_ns = (support_samples - 1) * sampling_period_ns

    # Delay for a causal implementation
    group_delay_ns = support_span_ns / 2

    # Deterministic difference caused by rounding to full samples
    rounding_error_ns = actual_window_ns - window_length_ns

    # Resolution scale if an arbitrary continuous window is rounded
    # to the nearest sample
    window_resolution_ns = sampling_period_ns / 2

    sensitivity = (
        n_smoothings * actual_window_ns / effective_length_ns
    )

    effective_resolution_ns = (
        sensitivity * window_resolution_ns
    )

    return {
        "window_samples": window_samples,
        "actual_window_ns": actual_window_ns,
        "effective_samples": effective_samples,
        "effective_length_ns": effective_length_ns,
        "support_samples": support_samples,
        "support_span_ns": support_span_ns,
        "group_delay_ns": group_delay_ns,
        "rounding_error_ns": rounding_error_ns,
        "effective_resolution_ns": effective_resolution_ns,
    }

In [ ]:
result = effective_smoothing_length(
    window_length_ns=500,
    n_smoothings=3,
    sampling_period_ns=10,
)
for name, value in result.items():
    print(f"{name}: {value}")

window_samples: 6
actual_window_ns: 60
effective_samples: 10.295630140987
effective_length_ns: 102.95630140987001
support_samples: 16
support_span_ns: 150
group_delay_ns: 75.0
rounding_error_ns: 0
effective_resolution_ns: 8.741572761215377
